# Searching a codebase: two questions, one model

A debugger shows you one execution path. `rg` shows you the whole codebase at once. They answer different questions.

This exercise teaches how to use `rg` to understand unfamiliar code — finding where a name comes from, how many files share a bug pattern, and why an error appears in a file that clearly has the right import.

**Prerequisite:** `importerror-fix-pattern` — this exercise builds on it.

**Structure:** encounter → scope → formulate → reveal → exercise. Run in order.

In [ ]:
import os, shutil

SRC = "/home/dev/sandbox/passagemath/src"

if not shutil.which("rg"):
    raise EnvironmentError("rg not found — install ripgrep: sudo apt install ripgrep")
if not os.path.isdir(SRC):
    raise EnvironmentError(f"Source tree not found: {SRC}\nUpdate SRC above to your passagemath clone.")

print(f"rg found, source tree at {SRC}")
# All !rg cells below use {SRC}. Run this cell first.

## Section 1: An error that shouldn't be there

CI logs show a failure in `ell_point.py`:

```
NameError: name 'pari' is not defined
```

That's strange. `ell_point.py` imports pari at the top of the file. How is the name undefined?

The obvious first move: search the file for `pari`.

In [ ]:
!rg '\bpari\b' {SRC}/sage/schemes/elliptic_curves/ell_point.py | head -20

`pari` appears 46 times. Import lines, docstrings, method bodies — all mixed together. This isn't useful.

The problem is the question. "Where does the word 'pari' appear?" is not the same as "where does `pari` get defined?"

## Section 2: Two questions

Every useful codebase search is one of:

- **"Where is X defined?"** — imports, assignments, function definitions
- **"What calls X?"** — usage sites, call expressions, references

They're different searches. A search for a bare name conflates them.

Start with the first one: where is `pari` defined in this file?

In [ ]:
!rg 'from sage\.libs\.pari import' -B1 -A5 {SRC}/sage/schemes/elliptic_curves/ell_point.py

The `except ImportError` block binds `PariError = ()`.

`pari` is not in the except block.

How many files have this same pattern?

In [ ]:
!rg 'from sage\.libs\.pari import' --type py -l {SRC}/

47 files. You didn't open any of them.

Now the second question: in `ell_point.py`, what actually *calls* `pari`?

There are two forms: `pari.method()` (attribute access) and `pari(args)` (direct call). The pattern `\bpari[.(]` finds both.

In [ ]:
!rg '\bpari[.(]' {SRC}/sage/schemes/elliptic_curves/ell_point.py

Calls throughout the file. If `pari` is unbound when any of those methods runs, the crash is guaranteed.

## Section 3: What "not bound" looks like

Here's the Python behavior that makes this a bug:

In [ ]:
# What happens when a try block fails before assignment?
try:
    raise ImportError("not installed")
    _fast = lambda n: n * 42   # never runs
except ImportError:
    pass

_fast(5)   # run me

In [ ]:
# What ell_point.py actually does: binds PariError but not pari.
try:
    raise ImportError("pari not available")
    _pari_demo = lambda n: n * 42   # never runs
    _PariError_demo = Exception     # never runs
except ImportError:
    _PariError_demo = ()            # bound — like PariError = ()

print("PariError fallback:", _PariError_demo)   # works
_pari_demo(5)                                   # crashes

Same except block. One name bound, one not.

The `NameError` tells you `pari` is undefined. It doesn't tell you `PariError` was bound.

You'd never know from the traceback that the except block did anything at all.

To see the asymmetry, you need the import block with context — which is what `-A5` gives you.

## Section 4: Three files

`ell_point.py` isn't the only one. Here are three files with the same import pattern and different except strategies — all of them missing `pari`.

In [ ]:
!rg 'from sage\.libs\.pari import' -B1 -A5 {SRC}/sage/schemes/elliptic_curves/ell_point.py

In [ ]:
!rg 'from sage\.libs\.pari import' -B1 -A5 {SRC}/sage/rings/number_field/number_field.py

In [ ]:
!rg 'from sage\.libs\.pari import' -B1 -A5 {SRC}/sage/rings/finite_rings/integer_mod_ring.py

Three files. Three except strategies — `PariError = ()`, `pari_gen = ()`, `class PariError(Exception): pass`. Zero bindings for `pari` itself.

Plot twist: the except block handles `PariError` carefully in every case — it's what the code catches exceptions with. `pari` gets nothing — it's what the code calls.

This is only visible with `-A5`. The error message won't tell you what the except block did bind.

---

**Pause.** Before continuing: what distinguishes a file with this bug from one without it?

Write your answer before reading Section 5.

---

## Section 5: The bug class

The signature: a file imports `pari` in a `try` block, calls `pari(...)` or `pari.something(...)` in methods, but the `except ImportError` block doesn't bind `pari`.

The method for finding instances:

1. **Scope**: `rg 'from sage\.libs\.pari import' --type py -l` — 47 candidate files
2. **Check each**: `rg 'from sage\.libs\.pari import' -B1 -A5` on each — shows the except block
3. **Flag**: if the except block doesn't bind `pari`, the file is affected

You've done step 2 three times. The exercise applies all three steps to a file you haven't diagnosed yet.

## Section 6: Exercise

`cusps.py` has the same import pattern. You haven't seen its except block yet.

**Your task:** answer three questions. For Q1 and Q2, write the `rg` search yourself.

1. Does `cusps.py` bind `pari` in its except block? (Apply step 2 of the method above.)
2. What method calls `pari` directly? Find the call site.
3. If `pari = None` were added to the except block, what would that method raise when `b` is non-zero? What about when `b` is zero? Why are the two cases different?

Write your searches and answers as comments before looking at the answer cell.

In [ ]:
# Q1: show the import block and except clause in cusps.py
# Hint: same pattern as Section 4, different file.
# cusps.py is at: src/sage/modular/cusps.py


In [ ]:
# Q2: find the method that calls pari directly
# Hint: use \bpari[.(] to catch both pari.method() and pari(args) forms
# Lines starting with 'sage:' are doctest examples — not real call sites
# cusps.py is at: src/sage/modular/cusps.py


In [ ]:
# Q3: test both cases
_pari_stub = None

# Case 1: pari(self.__a / b) — calling None as a function
try:
    _pari_stub(87)
except Exception as e:
    print("call:", type(e).__name__, "—", e)

# Case 2: pari.oo() — attribute access on None
try:
    _ = _pari_stub.oo
except Exception as e:
    print("attr:", type(e).__name__, "—", e)

### Answer

**Q1:** The except block in `cusps.py` binds `pari_gen = ()` — same as `number_field.py`. `pari` is not bound.

**Q2:** `Cusp.__pari__` contains: `return pari(self.__a / b) if b else pari.oo()`. Both branches call `pari` — one as a function, one as an attribute.

**Q3:** It depends on which branch runs.
- When `b` is non-zero: `pari(self.__a / b)` executes. `None(...)` raises `TypeError: 'NoneType' object is not callable`.
- When `b` is zero: `pari.oo()` executes. `None.oo` raises `AttributeError: 'NoneType' object has no attribute 'oo'`.

Same bug, same fix, two different error types depending on the argument. Cell 24 tests both.

**Fix commit:** `cb2fceaafec` (`cusps.py`, `integer_mod_ring.py`, `binary_qf.py`) — PR [#2282](https://github.com/passagemath/passagemath/pull/2282). The fix: `pari = None` in the except block, `if pari is None: sage__libs__pari().require()` at each call site. That part is in `importerror-fix-pattern`.

## Summary

Two questions:

- **"Where is X defined?"** — find imports, assignments, definitions. Use `rg 'from ... import X' -B1 -A5` to see what the except block does with the name.
- **"What calls X?"** — find call sites. Use `rg '\bX\.'` or `rg '\bX('` to find actual call expressions.

The `-A5` flag is the key to the first question. It shows the context that makes the asymmetry visible — which names the except block binds and which it doesn't.

To find a bug class in an unfamiliar codebase:
1. State the symptom as a text pattern
2. Ask "where is X defined?" to scope the candidate files
3. Ask "what calls X?" to find call sites
4. Use `-A5` on definition sites to check whether the fallback is complete

In [ ]:
# 47 candidate files. For each, check the except block with -B1 -A5.
# Files that don't bind pari need the fix from importerror-fix-pattern.
!rg 'from sage\.libs\.pari import' --type py -l {SRC}/